# Why are the new scores lower? Three checks for one region
1. **Same configs, old vs new split** - re-score the OLD study's best configs under the new polygon split. If they now score like the new frontier, the drop is the split, not the sampler.
2. **Random-pixel vs polygon split** - one config, one year, both ways. The gap is the size of the old leak.
3. **Per-class F1** - which classes pull the macro-F1 down (rare classes with ~1 validation polygon).

In [1]:
# Cell 1: Setup
import json
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import spatial_split as ss
from year_check import pick_configs, evaluate

REGION = 'DES'
OLD_JSON = f'data/old_results/optuna_regional_gee_configs/{REGION}_all_trials.json'   # the OLD study's all_trials.json (restore it from git; the new run overwrote the one in data/)
NEW_DIR = Path('data/optuna_regional_gee_configs')
N_CONFIGS = 6                                         # old configs to re-score
CHECK_YEARS = ss.search_years(REGION)[::3]            # 3 of the search years, to keep it quick

art = Path('data/feature_selection_artifacts')
all_ranked = json.load(open(art / f'{REGION}_ranked_features.json'))['ranked_features']
uncorr = json.load(open(art / f'{REGION}_uncorrelated_features.json'))['uncorrelated_features']
old_codes = [5, 3, 4, 12, 11, 66, 19, 36, 9, 41, 24, 23, 75, 30, 25, 33, 76, 34, 61]
new_codes = [5, 3, 4, 12, 11, 66, 19, 14, 14, 14, 22, 22, 22, 22, 22, 33, 76, 34, 61]
ref_df = pd.read_csv('data/classwise_samples_reference_ratios.csv'); ref_df['new_code'] = new_codes
ref_cols = [c for c in ref_df.columns if c not in ['l2_code', 'l2_name', 'l1_parent', 'old_code', 'new_code']]
ref_series = ref_df.groupby('new_code')[ref_cols].sum()[REGION]
print(REGION, 'check years:', CHECK_YEARS)

DES check years: [1991, 2002, 2016]


In [2]:
# Cell 2: CHECK 1 - old study's best configs scored on the NEW split
cfgs = pick_configs(OLD_JSON, n=N_CONFIGS)
res = evaluate(REGION, cfgs, CHECK_YEARS, all_ranked, uncorr, ref_series, seeds=(0,), n_jobs=4)
M = res.groupby('cfg').f1.mean()
tab = pd.DataFrame({'trial': [c['trial'] for c in cfgs], 'F1 in old study (leaky split)': [round(c['old_f1'], 3) for c in cfgs],
                    'F1 on new polygon split': M.round(3).values, 'complexity': ['%.1e' % c['complexity'] for c in cfgs]})
print(tab.to_string(index=False))
new_front = pd.read_json(NEW_DIR / f'{REGION}_pareto_frontier.json')
print(f"\nnew study's best frontier F1: {new_front.macro_f1.max():.3f}")

[DES] 1991 done
[DES] 2002 done
[DES] 2016 done
 trial  F1 in old study (leaky split)  F1 on new polygon split complexity
     1                          0.982                    0.520    1.1e+12
    13                          0.943                    0.503    1.0e+11
    16                          0.931                    0.448    2.6e+10
    28                          0.923                    0.465    7.0e+09
    30                          0.980                    0.508    1.6e+11
    31                          0.956                    0.520    1.3e+11
    37                          0.984                    0.530    2.0e+12
    39                          0.949                    0.488    1.2e+11

new study's best frontier F1: 0.554


In [3]:
# Cell 3: CHECK 2 - same config, one year: random-pixel split vs polygon split
yr = CHECK_YEARS[0]
sp = ss.load_year_split(REGION, yr, all_ranked, gated=True)
Xtr, ytr = sp['train']; Xva, yva = sp['val']
X, y = np.vstack([Xtr, Xva]), np.concatenate([ytr, yva])
rows = []
for c in cfgs[-3:]:
    names = uncorr if c['feature_strategy'] == 'uncorrelated' else all_ranked[:int(c['max_input_features'])]
    ix = [all_ranked.index(f) for f in names]
    mk = lambda: RandomForestClassifier(n_estimators=c['numberOfTrees'], min_samples_leaf=c['minLeafPopulation'],
                                        max_features='sqrt', n_jobs=4, random_state=0)
    poly = f1_score(yva, mk().fit(Xtr[:, ix], ytr).predict(Xva[:, ix]), average='macro')
    strat = y if np.bincount(np.unique(y, return_inverse=True)[1]).min() >= 2 else None
    a, b, ya, yb = train_test_split(X[:, ix], y, test_size=len(yva) / len(y), stratify=strat, random_state=0)
    rand = f1_score(yb, mk().fit(a, ya).predict(b), average='macro')
    rows.append(dict(trial=c['trial'], year=yr, random_pixel_split=round(rand, 3), polygon_split=round(poly, 3), gap=round(rand - poly, 3)))
print(pd.DataFrame(rows).to_string(index=False))

 trial  year  random_pixel_split  polygon_split   gap
    31  1991               0.985          0.468 0.517
    37  1991               0.989          0.472 0.518
    39  1991               0.977          0.457 0.520


In [4]:
# Cell 4: CHECK 3 - which classes drag macro-F1 down (best new trial, averaged over search years)
pm = json.load(open(NEW_DIR / f'{REGION}_per_year_metrics.json'))
best = max((t for t in pm['trials'] if t['values']), key=lambda t: t['values'][0])
labels = best['labels']; ys = best['per_year']
f1c = np.array([v['f1_per_class'] for v in ys.values()])
sup = np.array([np.array(v['confusion']).sum(1) for v in ys.values()])
out = pd.DataFrame({'class': labels, 'mean_F1': f1c.mean(0).round(3), 'min_year_F1': f1c.min(0).round(3),
                    'val_pixels_per_year': sup.mean(0).round(0).astype(int)}).sort_values('mean_F1')
print(f"best trial {best['trial']}  mean macro-F1 {best['values'][0]:.3f}")
print(out.to_string(index=False))
cr = [p for p in ss.SPLIT_DIR.glob(f'{REGION}_poly_*_class_report.csv')]
if cr: print('\nclass report (pixels per split over all years):'); print(pd.read_csv(cr[0]).to_string(index=False))

best trial 10  mean macro-F1 0.554
 class  mean_F1  min_year_F1  val_pixels_per_year
    76    0.035        0.000                 4206
    11    0.109        0.055                   94
    66    0.387        0.276                  373
    33    0.548        0.442                 1788
    14    0.563        0.244                   33
    61    0.637        0.148                 3089
    22    0.682        0.518                 2983
    12    0.715        0.645                 2426
     5    0.751        0.132                  170
    19    0.751        0.337                 4013
     4    0.913        0.869                 2242

class report (pixels per split over all years):
 class  train_px  val_px  test_px  train_share  val_share  test_share
     4    309545   77565     4792        0.790      0.198       0.012
     5    151760    6564    10194        0.901      0.039       0.060
    11    180218    3305     8535        0.938      0.017       0.044
    12    230727   82692    41519   